# 56 구조화 검색: DB와 조건 추출 원인 분리

55 사후 진단의 후속 개발 실험이다. 운영 코드·원본 DB·JSON·PDF는 변경하지 않는다.

56A: 같은 저장 조건과 수동 조건을 원본/복사 DB에서 실행한다. 명시 무실적, 교환/적립, 이용액/지급액 한도를 원문으로 구분한다. 변경 전건 원문 불일치·불확실0 및 독립 리뷰 후에만 56B를 실행한다.

56B: 원문 근거37질문 × 짧은/상세 × 기존/보완 추출 ×3회=444회. 동일 추출을 양 DB에 적용하므로 DB와 추출 효과를 분리한다. 질문별 짧은/상세는 외부 카드·발급사 범위를 동일하게 유지한다. 새 동의어·오타 보정은 없다.

공유 `.py`는 장시간 API의 요청별 원응답 저장·중단 재개·합산 비용 상한과 SQLite/JSON 버전 결속만 담당한다. 이 노트북은 실행 순서와 결과를 표시한다. HTML은 결과를 확인한 뒤 직접 작성한다. 지연은 parser/SQL/검색/답변 단계를 분리하며 이 실험에서는 parser/SQL만 측정한다.

In [1]:
from pathlib import Path
import importlib.util, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/scripts").is_dir())
spec = importlib.util.spec_from_file_location("runtime56_notebook", ROOT / "notebooks/scripts/56_structured_experiment_runtime.py")
R = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = R
spec.loader.exec_module(R)
print("기존 원본/복사 DB 및 중단 재개 규칙을 읽었습니다.")

기존 원본/복사 DB 및 중단 재개 규칙을 읽었습니다.


In [2]:
comparison_inputs = {str(path.relative_to(ROOT)): R.P.sha(path.read_bytes())
    for path in [R.DB1 / "structured.sqlite", R.DB1 / "facts.jsonl", R.DB2 / "structured.sqlite", R.DB2 / "facts.jsonl",
        ROOT / "notebooks/data/52_corrected_four_retrieval_modes/rerouted_repeat1_plans.json",
        ROOT / "notebooks/data/54_new_question_confirmation/parser_results.json"]}
comparison_contract_path = R.BASE / "db_comparison_input_contract.json"
if comparison_contract_path.exists():
    if R.read(comparison_contract_path) != comparison_inputs:
        raise RuntimeError("Stored56A DB/source inputs changed; inspect before replay")
    paired = R.read(R.BASE / "stored_plan_db_comparison.json")["records"]
    paired54 = R.read(R.BASE / "preserved54_db_comparison.json")["records"]
    probes = R.read(R.BASE / "manual_condition_db_comparison.json")["records"]
    print("동일 입력의56A 보존 결과 재사용:", len(paired), len(paired54), len(probes))
else:
    # 56A: preserve each original plan, changing only the paired DB/fact version.
    original_records = R.read(ROOT / "notebooks/data/52_corrected_four_retrieval_modes/rerouted_repeat1_plans.json")["records"]
    version_maps = [R.fact_map(R.DB1), R.fact_map(R.DB2)]
    paired = []
    for record in original_records:
        parser = record["parser"]
        routes = [R.route(parser["plan"], record["original_scope"], directory, facts=facts,
                          parser_failed=parser["status"] != "ok")
                  for directory, facts in zip([R.DB1, R.DB2], version_maps)]
        paired.append(dict(query_id=parser["query_id"], plan=parser["plan"],
                           original_scope=record["original_scope"], db_v1=routes[0], db_v2=routes[1]))
    R.write(R.BASE / "stored_plan_db_comparison.json", dict(records=paired))
    print("동일 저장 조건 비교:", len(paired), "건")
    print("카드 목록이 달라진 질문:", [r["query_id"] for r in paired if r["db_v1"]["eligible_cards"] != r["db_v2"]["eligible_cards"]])
    # Independent fact-level probes expose field fixes even when another row on the
    # same card keeps the aggregate SQL result unchanged.
    review = R.read(R.BASE / "db_source_review/normalization_source_review.json")
    probes = []
    for change in review["records"]:
        fid = change["fact_id"]
        old, new = [facts[fid] for facts in version_maps]
        kwargs = {}
        if "explicit_no_previous_month_requirement" in change["rules"]:
            kwargs["max_spend_krw"] = 0
        elif "exchange_voucher_is_not_accrual" in change["rules"]:
            kwargs["action"] = "earn"
        else:
            kwargs["min_cap_krw"] = old["derived"]["cap_krw"]
        requirement = R.Q.requirement([old["fields"]["target"]], **kwargs)
        probe_plan = R.Q.plan([requirement])
        routes = [R.query_database(probe_plan, [old["document_id"]], directory, facts=facts)
                  for directory, facts in zip([R.DB1, R.DB2], version_maps)]
        probes.append(dict(fact_id=fid, card_key=old["document_id"], rules=change["rules"],
            condition=requirement, fact_match_v1=R.Q.evaluate_fact(old,requirement),
            fact_match_v2=R.Q.evaluate_fact(new,requirement),
            sql_cards_v1=routes[0]["eligible_cards"], sql_cards_v2=routes[1]["eligible_cards"]))
    R.write(R.BASE / "manual_condition_db_comparison.json", dict(records=probes))
    print("수동 조건 검사:", len(probes))

    # Preserve the stopped54 parser set: it contains the no-spend and exchange
    # source-gate witnesses, but no previous generated-answer quality claim.
    old54 = R.read(ROOT / "notebooks/data/54_new_question_confirmation/parser_results.json")["records"]
    paired54 = []
    for record in old54:
        parser = record["parser"]
        routes = [R.route(parser["plan"], record["original_scope"], directory, facts=facts,
                          parser_failed=parser["status"] != "ok")
                  for directory, facts in zip([R.DB1, R.DB2], version_maps)]
        paired54.append(dict(identity=parser["identity"], plan=parser["plan"],
            original_scope=record["original_scope"], db_v1=routes[0], db_v2=routes[1]))
    R.write(R.BASE / "preserved54_db_comparison.json", dict(records=paired54))
    print("54 원본 조건 재현:", len(paired54), "건")
    print("카드 목록이 달라진 추출:", [r["identity"] for r in paired54 if r["db_v1"]["eligible_cards"] != r["db_v2"]["eligible_cards"]])

    R.write(comparison_contract_path,comparison_inputs)


동일 입력의56A 보존 결과 재사용: 40 96 21


In [3]:
prepared = R.prepare56(ROOT / "notebooks/56_structured_normalization.ipynb")
print("동결 질문:", len(prepared["queries"]), "추출 요청:", prepared["root"]["logical_parser_requests"])

동결 질문: 74 추출 요청: 444


In [ ]:
resources = R.run56(allow_network=True)
print(json.dumps(resources, ensure_ascii=False, indent=2))

## 판정과 한계

추출 조건 정답은 question_pairs_draft.json에 API 전 수동 기록한다. 요청 설명과 WHERE 조건, 명시 제약 미지원, DB 속성 불일치를 구분한다. 구조화 조회 결과가 없는 모든 경우가 추출 오류를 의미하지 않는다. 한 번의 조건이 두 DB에서 달라지는지를 DB 보완 효과로 본다. 전체 결과와 수동 검토 후 결과 보고서를 직접 작성한다. 반복은 같은 질문의 변동이며 독립 질문 수를 늘리지 않는다.